# Brain-to-Text Metrics v2 (Colab)

## Colab Setup

Run this setup cell once in Colab before the rest of the notebook. It installs the pushed branch that contains the Colab loaders and evaluation code.

In [ ]:
import sys
import subprocess

BRANCH = "new_text-to-brain_metrics"
REPO_URL = f"git+https://github.com/neurovlm/neurovlm.git@{BRANCH}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", REPO_URL])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "accelerate"])

# If any Hugging Face files/models are private, uncomment these two lines:
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
import os
import hashlib
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from neurovlm import NeuroVLM
from neurovlm.data import load_dataset, load_latent
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoModel, AutoTokenizer


In [ ]:
LLM_BACKEND = "huggingface"
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BERTSCORE_MODEL = "microsoft/deberta-xlarge-mnli"
MINILM_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MPNET_MODEL = "sentence-transformers/all-mpnet-base-v2"
B2T_RANDOM_EVIDENCE_N_SAMPLES = 25
B2T_RANDOM_EVIDENCE_SEED = 13

MAX_B2T = None        # full available networks, PubMed test, and NeuroVault sets
RUN_NETWORKS = True
RUN_PUBMED = True
RUN_NEUROVAULT = True
RUN_GENERATED_TEXT = False  # slow LLM/BERTScore generated-text evaluation

B2T_TOP_K = 5
B2T_SIM_THR = 0.35
B2T_DATASETS = ["llm_neuro_terms", "kg_mesh", "cogatlas"]

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

DRIVE_ROOT = '/content/drive/MyDrive/neurovlm_metrics'
OUTPUT_DIR = Path(DRIVE_ROOT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving metrics, JSON files, and plots to: {OUTPUT_DIR}")

In [ ]:
nvlm = NeuroVLM()
B2T_ENCODER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_st_model = SentenceTransformer(MINILM_MODEL, device=B2T_ENCODER_DEVICE)
_mpnet_model = SentenceTransformer(MPNET_MODEL, device=B2T_ENCODER_DEVICE)
print(f"Ready. Sentence encoders on {B2T_ENCODER_DEVICE}.")


In [ ]:
def load_network_test_set_labels_table():
    try:
        return load_dataset("network_test_set_labels"), "huggingface:neurovlm/embedded_text/network_test_set_labels.csv"
    except Exception as hf_error:
        candidates = [
            Path("network_test_set_labels.csv"),
            Path("docs/03_evaluation/network_test_set_labels.csv"),
            Path.cwd() / "network_test_set_labels.csv",
            Path.cwd() / "docs/03_evaluation/network_test_set_labels.csv",
        ]
        for candidate in candidates:
            if candidate.exists():
                return pd.read_csv(candidate), str(candidate.resolve())
        raise FileNotFoundError(
            "Could not load network_test_set_labels.csv from Hugging Face or local paths. "
            "Upload it to neurovlm/embedded_text as network_test_set_labels.csv. "
            f"Hugging Face error: {hf_error}"
        )


network_labels_df, NETWORK_TEST_SET_SOURCE = load_network_test_set_labels_table()
network_labels_df = network_labels_df[network_labels_df["network_key"] != "unknown"].copy()

# One row per canonical network, derived from the network test-set CSV.
network_info = (
    network_labels_df
    .sort_values(["network_key", "raw_network_label"])
    .groupby("network_key", as_index=False)
    .agg(
        display=("network_name", "first"),
        short_definition=("short_definition", "first"),
        long_definition=("long_definition", "first"),
        mapped_terms=("mapped_terms", "first"),
        raw_aliases=("raw_network_label", lambda s: "; ".join(sorted(set(map(str, s))))),
    )
)

DISPLAY_TO_KEY = dict(zip(network_info["display"], network_info["network_key"]))
KEY_TO_DISPLAY = dict(zip(network_info["network_key"], network_info["display"]))
SHORT_LABELS = dict(zip(network_info["display"], network_info["short_definition"]))
LONG_LABELS = dict(zip(network_info["display"], network_info["long_definition"]))

network_info.to_csv(OUTPUT_DIR / "b2t_network_label_metadata.csv", index=False)
network_info.to_json(OUTPUT_DIR / "b2t_network_label_metadata.json", orient="records", indent=2)
print(f"Loaded {len(network_info)} canonical network labels from {NETWORK_TEST_SET_SOURCE}")
display(network_info[["network_key", "display", "short_definition"]])

In [ ]:
def _normalize_expected_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()

all_net_latents = load_latent("networks_neuro")
network_label_rows = (
    network_labels_df
    .drop_duplicates("raw_network_label")
    .set_index("raw_network_label")
)

networks_data = {}
for atlas_name, atlas_latents in all_net_latents.items():
    if not hasattr(atlas_latents, "items"):
        continue
    for raw_label, latent in atlas_latents.items():
        if raw_label not in network_label_rows.index:
            continue
        row = network_label_rows.loc[raw_label]
        sample_name = f"{atlas_name}:{raw_label}"
        networks_data[sample_name] = {
            "latent": latent,
            "short_gt": _normalize_expected_text(row["short_definition"]),
            "long_gt": _normalize_expected_text(row["long_definition"]),
            "network_key": row["network_key"],
            "display": row["network_name"],
            "atlas": atlas_name,
            "raw_network_label": raw_label,
        }

network_samples_df = pd.DataFrame([
    {"sample": name, "network_key": d["network_key"], "display": d["display"], "atlas": d["atlas"], "raw_network_label": d["raw_network_label"]}
    for name, d in networks_data.items()
])
network_samples_df.to_csv(OUTPUT_DIR / "b2t_network_eval_samples.csv", index=False)
network_samples_df.to_json(OUTPUT_DIR / "b2t_network_eval_samples.json", orient="records", indent=2)
print(f"Networks loaded: {len(networks_data)} labeled maps across {len(all_net_latents)} atlases")
display(network_samples_df.head())

In [ ]:
# PubMed summaries dataset: use summary text as long-form ground truth.
# These summaries better match the desired B2T LLM output than full abstracts.
df_summaries = load_dataset("pubmed_summaries")
df_pubs = load_dataset("pubmed_text")

if "test" not in df_summaries.columns:
    raise ValueError("pubmed_summaries must include a boolean 'test' column for PubMed evaluation.")
df_test = df_summaries[df_summaries["test"].fillna(False).astype(bool)].reset_index(drop=True)

_pmid_col = "pmid" if "pmid" in df_test.columns else df_test.columns[0]
_summary_col = "summary" if "summary" in df_test.columns else "description"
_title_col = "title" if "title" in df_test.columns else ("name" if "name" in df_test.columns else None)

# Fill title from publications by PMID if summaries do not carry title/name.
_pub_pmid_col = "pmid" if "pmid" in df_pubs.columns else df_pubs.columns[0]
_pub_title_col = "name" if "name" in df_pubs.columns else ("title" if "title" in df_pubs.columns else None)
pub_title_lookup = {}
if _pub_title_col is not None:
    pub_title_lookup = (
        df_pubs
        .assign(_pmid_key=lambda df: df[_pub_pmid_col].astype(str))
        .drop_duplicates("_pmid_key")
        .set_index("_pmid_key")[_pub_title_col]
        .astype(str)
        .to_dict()
    )

pubmed_latents, pubmed_pmids = load_latent("pubmed_images")
pubmed_pmids = np.asarray(pubmed_pmids).astype(str)
test_pmids = df_test[_pmid_col].astype(str).values
mask = np.isin(pubmed_pmids, test_pmids)
aligned_latents = pubmed_latents[mask]
aligned_pmids = pubmed_pmids[mask]

pmid_to_row = (
    df_test
    .assign(_pmid_key=lambda df: df[_pmid_col].astype(str))
    .drop_duplicates("_pmid_key")
    .set_index("_pmid_key")
)

pubmed_data = []
for i, pmid in enumerate(aligned_pmids):
    if pmid not in pmid_to_row.index:
        continue
    row = pmid_to_row.loc[pmid]
    title = str(row[_title_col]) if _title_col is not None and _title_col in row.index else pub_title_lookup.get(pmid, "")
    summary = str(row[_summary_col]) if _summary_col in row.index else ""
    pubmed_data.append({
        "pmid": pmid,
        "latent": aligned_latents[i],
        "short_gt": title,
        "long_gt": summary,
    })

pubmed_eval = pubmed_data[:MAX_B2T] if MAX_B2T else pubmed_data
print(f"PubMed summary test samples: {len(pubmed_eval)} / {len(pubmed_data)}")
print(f"Summaries table rows: {len(df_summaries):,}; test rows: {len(df_test):,}")

In [ ]:
df_nv = load_dataset("neurovault_text")
df_nv_meta = load_dataset("neurovault_images_meta")
nv_latents = load_latent("neurovault_images")

_doi_pub = "doi" if "doi" in df_nv.columns else df_nv.columns[0]
_doi_meta = "doi" if "doi" in df_nv_meta.columns else df_nv_meta.columns[0]
_title_nv = "title" if "title" in df_nv.columns else df_nv.columns[1]
_abs_nv = "abstract" if "abstract" in df_nv.columns else df_nv.columns[2]

neurovault_data = []
for _, pub_row in df_nv.iterrows():
    doi = pub_row[_doi_pub]
    img_indices = np.where((df_nv_meta[_doi_meta] == doi).values)[0]
    if len(img_indices) == 0 or img_indices[0] >= len(nv_latents):
        continue
    neurovault_data.append({
        "doi": doi,
        "latent": nv_latents[int(img_indices[0])],
        "short_gt": str(pub_row[_title_nv]),
        "long_gt": str(pub_row[_abs_nv]),
    })

neurovault_eval = neurovault_data[:MAX_B2T] if MAX_B2T else neurovault_data
print(f"NeuroVault samples: {len(neurovault_eval)} / {len(neurovault_data)}")

## Metric Helpers

`nvlm_sim` is a cosine similarity in NeuroVLM's shared latent space. A value around 0.33 can be meaningful because it is measured against a broad retrieval space, so the notebook plots it with empirical quartiles and a random-pair baseline rather than treating 1.0 as the only intuitive target.

In [ ]:
def _semantic_sim(gen: str, gt: str) -> float:
    emb1 = _st_model.encode(gen, convert_to_tensor=True)
    emb2 = _st_model.encode(gt, convert_to_tensor=True)
    return float(st_util.cos_sim(emb1, emb2))


def _bertscore_single(generated: str, reference: str):
    p, r, f1 = bert_score(
        cands=[generated],
        refs=[reference],
        lang="en",
        model_type=BERTSCORE_MODEL,
        verbose=False,
    )
    return float(p[0]), float(r[0]), float(f1[0])


def _nvlm_latent_sim(brain_query_emb: torch.Tensor, generated: str) -> float:
    nvlm._ensure_projection_heads()
    with torch.no_grad():
        raw_emb = nvlm._encode_text(generated)
        z_text = nvlm._proj_head_text_infonce(raw_emb.to(nvlm.device))
        z_text = F.normalize(z_text, dim=-1).cpu()
    z_brain = brain_query_emb.cpu()
    if z_brain.dim() == 1:
        z_brain = z_brain.unsqueeze(0)
    return float(F.cosine_similarity(z_brain, z_text))


def _format_context_summary(table):
    lines = []
    for _, row in table.iterrows():
        lines.append(f"[{row.get('dataset', '?')}] sim={row.get('cosine_similarity', float('nan')):.3f} | {row.get('title', '')}")
    return "\n".join(lines)


def _b2t_once(table, user_prompt, gt_text, max_tokens, brain_query_emb):
    generated = nvlm.generate_llm_response(
        backend=LLM_BACKEND,
        model_name=LLM_MODEL,
        table=table,
        user_prompt=user_prompt,
        max_new_tokens=max_tokens,
        verbose=False,
    )
    bert_p, bert_r, bert_f1 = _bertscore_single(generated, gt_text)
    return {
        "generated": generated,
        "gt_text": gt_text,
        "bert_p": bert_p,
        "bert_r": bert_r,
        "bert_f1": bert_f1,
        "sem_sim": _semantic_sim(generated, gt_text),
        "nvlm_sim": _nvlm_latent_sim(brain_query_emb, generated),
    }


def run_b2t(name, latent, short_gt, long_gt, short_prompt, long_prompt="", short_tokens=64, long_tokens=512, datasets=None):
    try:
        result = nvlm.brain(latent).to_text(datasets=datasets or B2T_DATASETS)
        all_table = result.top_k(B2T_TOP_K)
        table = all_table[all_table["cosine_similarity"] > B2T_SIM_THR]
        if table.empty:
            table = all_table
        if len(table) > B2T_TOP_K:
            table = table.nlargest(B2T_TOP_K, "cosine_similarity").reset_index(drop=True)

        records = []
        for mode, prompt, gt, tokens in [
            ("short", short_prompt, short_gt, short_tokens),
            ("long", long_prompt, long_gt, long_tokens),
        ]:
            rec = _b2t_once(table, prompt, gt, tokens, result.query_embeddings)
            rec.update({"name": name, "mode": mode, "context_summary": _format_context_summary(table)})
            records.append(rec)
        return records
    except Exception as e:
        print(f"[B2T error] {name}: {type(e).__name__}: {e}")
        traceback.print_exc()
        return []


SHORT_PROMPT_GENERAL = (
    "Reply with a single concise sentence (10-20 words) naming the main cognitive "
    "function or brain network. Output only that sentence."
)
SHORT_PROMPT_PUBMED = (
    "Generate only a paper title (6-12 words) for the neuroimaging study this "
    "brain activation pattern represents. Output the title only."
)
LONG_PROMPT = ""

In [ ]:
def normalize_label_text(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def _network_aliases(row):
    aliases = []
    for value in [row["display"], row.get("mapped_terms", ""), row.get("raw_aliases", "")]:
        aliases.extend([normalize_label_text(x) for x in str(value).split(";")])
    aliases.append(normalize_label_text(row["network_key"].replace("_", " ")))
    return [x for x in dict.fromkeys(aliases) if x]


NETWORK_LABEL_ROWS = network_info.to_dict("records")
NETWORK_ALIAS_MAP = {row["network_key"]: _network_aliases(row) for row in NETWORK_LABEL_ROWS}


def predict_network_label(text: str, min_semantic_margin=0.02):
    text_norm = normalize_label_text(text)
    alias_hits = []
    for key, aliases in NETWORK_ALIAS_MAP.items():
        for alias in aliases:
            if alias and re.search(rf"\b{re.escape(alias)}\b", text_norm):
                alias_hits.append((key, alias))
                break
    if len(alias_hits) == 1:
        return alias_hits[0][0], "alias", alias_hits[0][1], 1.0

    label_texts = [f"{row['display']}. {row['long_definition']}" for row in NETWORK_LABEL_ROWS]
    generated_emb = _st_model.encode(text, convert_to_tensor=True)
    label_emb = _st_model.encode(label_texts, convert_to_tensor=True)
    sims = st_util.cos_sim(generated_emb, label_emb).cpu().numpy().ravel()
    order = sims.argsort()[::-1]
    keys = [row["network_key"] for row in NETWORK_LABEL_ROWS]
    margin = float(sims[order[0]] - sims[order[1]]) if len(order) > 1 else float("nan")
    method = "semantic" if margin >= min_semantic_margin else "semantic_low_margin"
    best_row = NETWORK_LABEL_ROWS[order[0]]
    return keys[order[0]], method, best_row["display"], float(sims[order[0]])


def add_network_label_accuracy(df):
    out = df.copy()
    preds = out["generated"].apply(predict_network_label)
    out["pred_network_key"] = [p[0] for p in preds]
    out["label_match_method"] = [p[1] for p in preds]
    out["label_match_evidence"] = [p[2] for p in preds]
    out["label_match_score"] = [p[3] for p in preds]
    out["true_network_key"] = out["name"].map(lambda name: networks_data[name]["network_key"])
    out["network_label_correct"] = out["pred_network_key"] == out["true_network_key"]
    return out

## Retrieval Evidence Metrics

These analyses evaluate the retrieval stage before any LLM generation. That keeps the metric tied to what the brain encoder ranks, rather than to how well the LLM writes from those ranked terms.

- **Approach 1: Gold-term ranking** asks whether known gold terms appear high in the ranked retrieval list. This is an exact term-recovery test for datasets with gold terms: Networks and PubMed.
- **Approach 3: Ground-truth text vs retrieved evidence** asks whether the retrieved terms and definitions collectively describe the same content as the reference text. This can run on Networks, PubMed, and NeuroVault.

Generated text evaluation is kept later and is disabled by default because it is much slower.


### MeSH Node-Type Filter

PubMed gold labels are MeSH terms, but not every MeSH category is inferable from a brain map. The default PubMed retrieval corpus excludes molecular terms. Inspect this cell and set `INCLUDE_MOLECULAR_MESH = True` if you want molecular terms included.

In [ ]:
INCLUDE_MOLECULAR_MESH = False
MESH_BRAIN_RANKABLE_NODE_TYPES = [
    "disorder",
    "anatomical_region",
    "biological_process",
    "cognitive_construct",
]
if INCLUDE_MOLECULAR_MESH:
    MESH_BRAIN_RANKABLE_NODE_TYPES.append("molecular")

PUBMED_B2T_DATASET = "kg_mesh_brain_rankable_plus_molecular" if INCLUDE_MOLECULAR_MESH else "kg_mesh_brain_rankable"
print(f"PubMed B2T will retrieve from: {PUBMED_B2T_DATASET}")
print(f"Allowed PubMed MeSH node types: {MESH_BRAIN_RANKABLE_NODE_TYPES}")

mesh_nodes = load_dataset("mesh_kg_nodes")
if "node_type" in mesh_nodes.columns:
    mesh_node_type_counts = mesh_nodes["node_type"].value_counts().rename("n_terms")
    mesh_node_type_counts.to_csv(OUTPUT_DIR / "b2t_mesh_node_type_counts.csv")
    mesh_node_type_counts.reset_index().rename(columns={"index": "node_type"}).to_json(OUTPUT_DIR / "b2t_mesh_node_type_counts.json", orient="records", indent=2)
    display(mesh_node_type_counts)
    molecular_examples = mesh_nodes[mesh_nodes["node_type"] == "molecular"].head(50)
    molecular_examples.to_csv(OUTPUT_DIR / "b2t_mesh_molecular_examples.csv", index=False)
    molecular_examples.to_json(OUTPUT_DIR / "b2t_mesh_molecular_examples.json", orient="records", indent=2)
    display(molecular_examples)
else:
    print("mesh_kg_nodes is missing node_type; PubMed node-type filtering cannot run.")

In [ ]:
TERM_EVAL_NORMALIZED_KS = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10]
TERM_RECALL_CURVE_NORMALIZED_KS = np.linspace(0.0, 1.0, 101)
B2T_TERM_EXAMPLE_TOP_K = 20
B2T_TERM_TOP_K = B2T_TERM_EXAMPLE_TOP_K
B2T_EVIDENCE_TOP_K = B2T_TOP_K
NETWORK_B2T_TERM_DATASETS = ["llm_neuro_terms", "kg_mesh", "cogatlas"]
B2T_TERM_DATASETS_BY_EVAL_DATASET = {
    "networks": NETWORK_B2T_TERM_DATASETS,
    "pubmed": [PUBMED_B2T_DATASET],
    "neurovault": NETWORK_B2T_TERM_DATASETS,
}
B2T_RETRIEVAL_TABLE_CACHE = {}


def normalize_term_text(text: str) -> str:
    text = str(text or "").lower()
    text = text.split("/")[0]
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def split_gold_terms(value) -> list[str]:
    if pd.isna(value):
        return []
    terms = []
    for chunk in re.split(r";|\n|\|", str(value)):
        term = chunk.strip()
        if term:
            terms.append(term)
    return terms


def terms_for_dataset(dataset_name: str) -> set[str]:
    df = load_dataset(dataset_name)
    if not isinstance(df, pd.DataFrame):
        return set()
    for col in ["term", "title", "name", "label"]:
        if col in df.columns:
            return {normalize_term_text(x) for x in df[col].dropna().astype(str)}
    return set()


def network_gold_terms(sample_name: str) -> list[str]:
    d = networks_data[sample_name]
    label_rows = network_labels_df[network_labels_df["raw_network_label"].astype(str) == str(d["raw_network_label"])]
    if label_rows.empty:
        label_rows = network_labels_df[network_labels_df["network_key"] == d["network_key"]]
    terms = []
    for col in ["mapped_terms", "region_terms", "cognitive_terms"]:
        if col in label_rows.columns:
            for value in label_rows[col].dropna().tolist():
                terms.extend(split_gold_terms(value))
    return list(dict.fromkeys(terms))


def load_pubmed_mesh_annotations_or_none():
    try:
        annotations = load_dataset("pubmed_mesh_annotations")
        print(f"Loaded PubMed MeSH annotations for {len(annotations):,} PMIDs")
        return annotations
    except Exception as e:
        print("PubMed MeSH gold annotations are unavailable; PubMed exact term-ranking metrics will be skipped.")
        print("Upload mesh_annotations.json to neurovlm/mesh_kg to enable them.")
        print(f"Loader error: {type(e).__name__}: {e}")
        return None


PUBMED_MESH_ANNOTATIONS = load_pubmed_mesh_annotations_or_none()
MESH_NODE_TYPE_BY_TERM = {}
if "node_type" in mesh_nodes.columns:
    name_col = "name" if "name" in mesh_nodes.columns else "term"
    MESH_NODE_TYPE_BY_TERM = {
        normalize_term_text(row[name_col]): row["node_type"]
        for _, row in mesh_nodes.iterrows()
        if pd.notna(row.get(name_col)) and pd.notna(row.get("node_type"))
    }


def pubmed_gold_terms(pmid) -> list[str]:
    if PUBMED_MESH_ANNOTATIONS is None:
        return []
    raw_terms = PUBMED_MESH_ANNOTATIONS.get(str(pmid), [])
    filtered = []
    for term in raw_terms:
        base = str(term).split("/")[0].strip()
        norm = normalize_term_text(base)
        if norm and MESH_NODE_TYPE_BY_TERM.get(norm) in set(MESH_BRAIN_RANKABLE_NODE_TYPES):
            filtered.append(base)
    return list(dict.fromkeys(filtered))


def table_terms(table: pd.DataFrame) -> list[str]:
    if table is None or table.empty:
        return []
    return [str(x) for x in table["title"].fillna("").tolist() if str(x).strip()]


def retrieval_table_for_sample(latent, dataset_name: str, sample: str):
    """Top evidence table for LLM context and Approach 3 evidence similarity."""
    cache_key = (dataset_name, str(sample), "evidence")
    if cache_key in B2T_RETRIEVAL_TABLE_CACHE:
        return B2T_RETRIEVAL_TABLE_CACHE[cache_key]
    datasets = B2T_TERM_DATASETS_BY_EVAL_DATASET[dataset_name]
    result = nvlm.brain(latent).to_text(datasets=datasets)
    all_table = result.top_k(max(B2T_TERM_TOP_K, B2T_EVIDENCE_TOP_K))
    table = all_table[all_table["cosine_similarity"] > B2T_SIM_THR]
    if table.empty:
        table = all_table
    table = table.nlargest(max(B2T_TERM_TOP_K, B2T_EVIDENCE_TOP_K), "cosine_similarity").reset_index(drop=True)
    B2T_RETRIEVAL_TABLE_CACHE[cache_key] = table
    return table


def full_retrieval_table_for_sample(latent, dataset_name: str, sample: str | None = None):
    """Full ranked retrieval table for normalized-k term recall metrics and random baselines."""
    cache_key = (dataset_name, str(sample), "full") if sample is not None else None
    if cache_key is not None and cache_key in B2T_RETRIEVAL_TABLE_CACHE:
        return B2T_RETRIEVAL_TABLE_CACHE[cache_key]
    datasets = B2T_TERM_DATASETS_BY_EVAL_DATASET[dataset_name]
    result = nvlm.brain(latent).to_text(datasets=datasets)
    n_candidates = sum(int(scores.shape[0]) for scores in result.scores_by_dataset.values())
    table = result.top_k(n_candidates)
    if cache_key is not None:
        B2T_RETRIEVAL_TABLE_CACHE[cache_key] = table
    return table


def unique_ranked_terms_from_table(table: pd.DataFrame) -> list[str]:
    if table is None or table.empty:
        return []
    ranked = table.sort_values("cosine_similarity", ascending=False, kind="mergesort").copy()
    ranked["_normalized_term"] = ranked["title"].map(normalize_term_text)
    ranked = ranked[ranked["_normalized_term"] != ""]
    ranked = ranked.drop_duplicates("_normalized_term", keep="first")
    return ranked["title"].astype(str).tolist()


_pub_abs_col = "description" if "description" in df_pubs.columns else ("abstract" if "abstract" in df_pubs.columns else None)
_pub_abs_lookup = {}
if _pub_abs_col is not None:
    _pub_abs_lookup = (
        df_pubs
        .assign(_pmid_key=lambda df: df[_pub_pmid_col].astype(str))
        .drop_duplicates("_pmid_key")
        .set_index("_pmid_key")[_pub_abs_col]
        .astype(str)
        .to_dict()
    )


def dataset_records_for_retrieval_eval():
    if RUN_NETWORKS:
        for name, d in networks_data.items():
            yield "networks", name, d["latent"], network_gold_terms(name), {"long_description": d["long_gt"]}
    if RUN_PUBMED:
        for d in pubmed_eval:
            pmid = str(d["pmid"])
            references = {"summary": d["long_gt"]}
            abstract = _pub_abs_lookup.get(pmid, "")
            if abstract:
                references["abstract"] = abstract
            yield "pubmed", pmid, d["latent"], pubmed_gold_terms(pmid), references
    if RUN_NEUROVAULT:
        for d in neurovault_eval:
            yield "neurovault", str(d["doi"]), d["latent"], [], {"abstract": d["long_gt"]}


### Approach 1: Gold-Term Ranking

This evaluates whether the full ranked term list retrieved from the brain includes known gold terms. Metrics use a normalized rank budget, `normalized_k = k / n_candidate_terms`, so PubMed and Networks are comparable even though their retrieval corpora have different sizes. It runs for Networks and PubMed only; NeuroVault is skipped because we do not have gold terms for it.

Metric columns:

- `normalized_k_target`: the requested fraction of the candidate corpus to inspect, such as `0.01` for the top 1%.
- `k`: the actual number of retrieved terms inspected, computed as `ceil(normalized_k_target * n_candidate_terms)`.
- `n_candidate_terms`: the number of unique terms the model could rank for that dataset after normalizing/deduplicating term names.
- `n_gold_terms`: gold terms that are reachable in the candidate corpus. Unreachable terms are excluded so the model is not penalized for terms it could never retrieve.
- `n_unreachable_gold_terms`: gold terms missing from the retrieval corpora; this is a fairness/audit check.
- `precision_at_normalized_k`: among the inspected top-k terms, the fraction that are gold terms. This checks how concentrated the retrieved list is with correct gold terms.
- `recall_at_normalized_k`: among the reachable gold terms, the fraction recovered within the inspected top-k terms. This is the main term-recovery metric.
- `hit_at_normalized_k`: whether at least one gold term appears in the inspected top-k terms. This is useful when each sample has few gold terms.
- `mrr_at_normalized_k`: reciprocal rank of the first gold-term hit if it appears within top-k. This rewards putting any correct term very high.
- `normalized_first_hit_rank`: first gold-term rank divided by `n_candidate_terms`; lower is better and accounts for corpus size.
- `expected_random_recall_at_normalized_k`: chance-level recall if terms were randomly ordered. For normalized k, this is approximately `normalized_k_target`.
- `recall_auc`: area under the recall-vs-normalized-k curve. Higher means gold terms appear earlier across the full ranked list.
- `expected_random_recall_auc`: chance AUC for the diagonal random baseline, `0.5`.
- `recall_auc_minus_random`: how much the retrieval curve improves over random ranking.

The recall curve plots `recall_at_normalized_k` on the y-axis and normalized k on the x-axis. A curve above the diagonal means the brain-to-text retrieval ranks gold terms earlier than random chance.


In [ ]:
def _k_from_normalized_k(normalized_k: float, n_candidates: int) -> int:
    if n_candidates <= 0 or normalized_k <= 0:
        return 0
    return min(n_candidates, max(1, int(np.ceil(float(normalized_k) * n_candidates))))


def _auc_trapezoid(x, y) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x))
    return float(np.trapz(y, x))


def exact_term_ranking_outputs(
    dataset: str,
    sample: str,
    gold_terms: list[str],
    retrieved_terms: list[str],
    reachable_terms: set[str] | None = None,
):
    gold_norm_all = {normalize_term_text(t) for t in gold_terms if normalize_term_text(t)}
    if reachable_terms is None:
        gold_norm = gold_norm_all
        excluded = set()
    else:
        gold_norm = gold_norm_all & reachable_terms
        excluded = gold_norm_all - reachable_terms

    retrieved_norm = []
    seen = set()
    for term in retrieved_terms:
        norm = normalize_term_text(term)
        if norm and norm not in seen:
            retrieved_norm.append(norm)
            seen.add(norm)

    if not gold_norm or not retrieved_norm:
        return [], [], None

    n_candidates = len(retrieved_norm)
    first_hit_rank = next((i + 1 for i, term in enumerate(retrieved_norm) if term in gold_norm), np.nan)
    normalized_first_hit_rank = float(first_hit_rank / n_candidates) if not pd.isna(first_hit_rank) else np.nan

    metric_rows = []
    for normalized_k_target in TERM_EVAL_NORMALIZED_KS:
        k = _k_from_normalized_k(normalized_k_target, n_candidates)
        topk = retrieved_norm[:k]
        hits = set(topk) & gold_norm
        normalized_k = k / n_candidates if n_candidates else np.nan
        metric_rows.append({
            "dataset": dataset,
            "sample": sample,
            "normalized_k_target": float(normalized_k_target),
            "normalized_k": float(normalized_k),
            "k": int(k),
            "n_candidate_terms": int(n_candidates),
            "n_gold_terms": len(gold_norm),
            "n_unreachable_gold_terms": len(excluded),
            "n_retrieved_terms": len(topk),
            "n_hits": len(hits),
            "precision_at_normalized_k": len(hits) / max(len(topk), 1),
            "recall_at_normalized_k": len(hits) / len(gold_norm),
            "hit_at_normalized_k": bool(hits),
            "mrr_at_normalized_k": 0.0 if pd.isna(first_hit_rank) or first_hit_rank > k else 1.0 / float(first_hit_rank),
            "normalized_first_hit_rank": normalized_first_hit_rank,
            "expected_random_recall_at_normalized_k": float(normalized_k_target),
            "matched_terms": "; ".join(sorted(hits)),
        })

    curve_rows = []
    recall_values = []
    for normalized_k_target in TERM_RECALL_CURVE_NORMALIZED_KS:
        k = _k_from_normalized_k(normalized_k_target, n_candidates)
        topk = retrieved_norm[:k]
        hits = set(topk) & gold_norm
        recall = len(hits) / len(gold_norm)
        recall_values.append(recall)
        curve_rows.append({
            "dataset": dataset,
            "sample": sample,
            "normalized_k_target": float(normalized_k_target),
            "normalized_k": float(k / n_candidates) if n_candidates else np.nan,
            "k": int(k),
            "n_candidate_terms": int(n_candidates),
            "n_gold_terms": len(gold_norm),
            "recall_at_normalized_k": float(recall),
            "expected_random_recall_at_normalized_k": float(normalized_k_target),
        })

    auc = _auc_trapezoid(TERM_RECALL_CURVE_NORMALIZED_KS, recall_values)
    auc_row = {
        "dataset": dataset,
        "sample": sample,
        "n_candidate_terms": int(n_candidates),
        "n_gold_terms": len(gold_norm),
        "n_unreachable_gold_terms": len(excluded),
        "recall_auc": float(auc),
        "expected_random_recall_auc": 0.5,
        "recall_auc_minus_random": float(auc - 0.5),
        "normalized_first_hit_rank": normalized_first_hit_rank,
    }
    return metric_rows, curve_rows, auc_row


network_candidate_terms = set().union(*(terms_for_dataset(ds) for ds in NETWORK_B2T_TERM_DATASETS))
network_missing_rows = []
for sample_name in networks_data:
    for term in network_gold_terms(sample_name):
        norm = normalize_term_text(term)
        if norm and norm not in network_candidate_terms:
            network_missing_rows.append({"sample": sample_name, "term": term, "normalized_term": norm})
network_missing_terms_df = pd.DataFrame(network_missing_rows).drop_duplicates() if network_missing_rows else pd.DataFrame(columns=["sample", "term", "normalized_term"])
network_missing_terms_df.to_csv(OUTPUT_DIR / "network_gold_terms_missing_from_retrieval_corpora.csv", index=False)
network_missing_terms_df.to_json(OUTPUT_DIR / "network_gold_terms_missing_from_retrieval_corpora.json", orient="records", indent=2)
display(network_missing_terms_df.head(25))
print(f"Network gold terms missing from llm_neuro_terms + kg_mesh + cogatlas: {len(network_missing_terms_df):,}")

pubmed_candidate_terms = terms_for_dataset(PUBMED_B2T_DATASET)


In [ ]:
term_metric_rows = []
term_curve_rows = []
term_auc_rows = []
term_examples = []

for dataset, sample, latent, gold_terms, _references in tqdm(list(dataset_records_for_retrieval_eval()), desc="Approach 1 normalized gold-term ranking"):
    if dataset == "neurovault":
        continue
    table = full_retrieval_table_for_sample(latent, dataset, sample)
    retrieved_terms = unique_ranked_terms_from_table(table)
    reachable_terms = network_candidate_terms if dataset == "networks" else pubmed_candidate_terms
    metric_rows, curve_rows, auc_row = exact_term_ranking_outputs(dataset, sample, gold_terms, retrieved_terms, reachable_terms=reachable_terms)
    term_metric_rows.extend(metric_rows)
    term_curve_rows.extend(curve_rows)
    if auc_row is not None:
        term_auc_rows.append(auc_row)
    term_examples.append({
        "dataset": dataset,
        "sample": sample,
        "gold_terms": "; ".join(gold_terms[:50]),
        "n_ranked_candidate_terms": len(retrieved_terms),
        "top_terms": "; ".join(retrieved_terms[:B2T_TERM_EXAMPLE_TOP_K]),
    })

b2t_term_metrics_df = pd.DataFrame(term_metric_rows)
b2t_term_recall_curve_df = pd.DataFrame(term_curve_rows)
b2t_term_auc_df = pd.DataFrame(term_auc_rows)
b2t_term_examples_df = pd.DataFrame(term_examples)

b2t_term_metrics_df.to_csv(OUTPUT_DIR / "b2t_approach1_gold_term_ranking_metrics.csv", index=False)
b2t_term_metrics_df.to_json(OUTPUT_DIR / "b2t_approach1_gold_term_ranking_metrics.json", orient="records", indent=2)
b2t_term_recall_curve_df.to_csv(OUTPUT_DIR / "b2t_approach1_gold_term_recall_curve.csv", index=False)
b2t_term_recall_curve_df.to_json(OUTPUT_DIR / "b2t_approach1_gold_term_recall_curve.json", orient="records", indent=2)
b2t_term_auc_df.to_csv(OUTPUT_DIR / "b2t_approach1_gold_term_recall_auc.csv", index=False)
b2t_term_auc_df.to_json(OUTPUT_DIR / "b2t_approach1_gold_term_recall_auc.json", orient="records", indent=2)
b2t_term_examples_df.to_csv(OUTPUT_DIR / "b2t_approach1_gold_term_examples.csv", index=False)
b2t_term_examples_df.to_json(OUTPUT_DIR / "b2t_approach1_gold_term_examples.json", orient="records", indent=2)

if len(b2t_term_metrics_df):
    b2t_term_metrics_summary_df = (
        b2t_term_metrics_df
        .groupby(["dataset", "normalized_k_target"])[
            [
                "normalized_k", "k", "n_candidate_terms",
                "precision_at_normalized_k", "recall_at_normalized_k",
                "hit_at_normalized_k", "mrr_at_normalized_k",
                "expected_random_recall_at_normalized_k", "n_unreachable_gold_terms",
            ]
        ]
        .mean()
        .round(3)
    )
    b2t_term_auc_summary_df = b2t_term_auc_df.groupby("dataset")[["recall_auc", "expected_random_recall_auc", "recall_auc_minus_random"]].agg(["mean", "std", "count"]).round(3)
    b2t_term_metrics_summary_df.to_csv(OUTPUT_DIR / "b2t_approach1_gold_term_ranking_summary.csv")
    b2t_term_metrics_summary_df.reset_index().to_json(OUTPUT_DIR / "b2t_approach1_gold_term_ranking_summary.json", orient="records", indent=2)
    b2t_term_auc_summary_df.to_csv(OUTPUT_DIR / "b2t_approach1_gold_term_recall_auc_summary.csv")
    b2t_term_auc_summary_df.reset_index().to_json(OUTPUT_DIR / "b2t_approach1_gold_term_recall_auc_summary.json", orient="records", indent=2)
    display(b2t_term_metrics_summary_df)
    display(b2t_term_auc_summary_df)
else:
    print("No exact gold-term metrics were computed. PubMed needs mesh_annotations.json; NeuroVault has no gold terms.")


In [ ]:
if len(b2t_term_recall_curve_df):
    fig, ax = plt.subplots(figsize=(7, 5))
    curve_summary = (
        b2t_term_recall_curve_df
        .groupby(["dataset", "normalized_k_target"])
        .agg(recall_mean=("recall_at_normalized_k", "mean"), recall_std=("recall_at_normalized_k", "std"), n=("sample", "count"))
        .reset_index()
    )
    auc_means = b2t_term_auc_df.groupby("dataset")["recall_auc"].mean().to_dict()
    for dataset, sub in curve_summary.groupby("dataset"):
        label = f"{dataset} (AUC={auc_means.get(dataset, np.nan):.2f})"
        ax.plot(sub["normalized_k_target"], sub["recall_mean"], label=label)
    ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random chance")
    ax.set_xlabel("Normalized k (k / n candidate terms)")
    ax.set_ylabel("Recall@k")
    ax.set_title("Approach 1: Normalized Gold-Term Recall Curve")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_approach1_normalized_recall_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No recall curve rows to plot.")


### Approach 3: Ground-Truth Text vs Retrieved Evidence

This evaluates whether the top retrieved terms and definitions collectively resemble the ground-truth text. It does not require exact gold terms, so it runs for Networks, PubMed, and NeuroVault.

Reference text used:

- Networks: long network description.
- PubMed: summary and abstract when available.
- NeuroVault: abstract.

Metric columns:

- `*_evidence_similarity`: cosine similarity between the reference text and the retrieved evidence text. The retrieved evidence text is built from the top retrieved term titles and descriptions.
- `minilm_evidence_similarity`: a lightweight general semantic-similarity score using `all-MiniLM-L6-v2`. This is a broad sanity check, not domain-specific.
- `mpnet_evidence_similarity`: a stronger general sentence-embedding score using `all-mpnet-base-v2`. This helps show that the result is not only high under SPECTER.
- `specter_evidence_similarity`: a scientific-literature-oriented score using the same SPECTER2 ad-hoc query style used by NeuroVLM. This is the primary domain-aligned text-similarity metric.
- `random_*_evidence_similarity_mean`: expected score by random chance, computed by repeatedly sampling random evidence sets of the same size from the same candidate corpus.
- `random_*_evidence_similarity_std`: variability of that random baseline across random evidence samples.
- `*_evidence_similarity_minus_random`: actual retrieved-evidence score minus the random baseline mean. This is the most important column for asking whether retrieval evidence is better than chance.
- `random_baseline_n_samples`: how many random evidence sets were sampled for each sample/reference pair.

Interpretation: high absolute similarity is useful, but the clearest evidence is a positive `*_minus_random` value across datasets and references. That means the model's ranked evidence is closer to the ground truth than random terms from the same retrieval corpus.


In [ ]:
def evidence_text(table: pd.DataFrame, k: int = B2T_EVIDENCE_TOP_K) -> str:
    lines = []
    for _, row in table.head(k).iterrows():
        title = str(row.get("title", "")).strip()
        desc = str(row.get("description", "")).strip()
        if title or desc:
            lines.append(f"{title}. {desc}".strip())
    return "\n".join(lines)


def random_evidence_text(table: pd.DataFrame, rng: np.random.Generator, k: int = B2T_EVIDENCE_TOP_K) -> str:
    if table is None or table.empty:
        return ""
    sample_n = min(int(k), len(table))
    sampled = table.sample(n=sample_n, replace=False, random_state=int(rng.integers(0, 2**32 - 1)))
    return evidence_text(sampled, k=sample_n)


def random_evidence_texts(table: pd.DataFrame, rng: np.random.Generator, n_samples: int = B2T_RANDOM_EVIDENCE_N_SAMPLES) -> list[str]:
    return [random_evidence_text(table, rng) for _ in range(n_samples)]


def sentence_cosines_to_reference(ref_text: str, candidate_texts: list[str], model: SentenceTransformer) -> list[float]:
    if not str(ref_text or "").strip():
        return [np.nan for _ in candidate_texts]
    cleaned = [str(text or "") for text in candidate_texts]
    if not cleaned:
        return []
    valid = [bool(text.strip()) for text in cleaned]
    scores = [np.nan for _ in cleaned]
    if not any(valid):
        return scores
    texts_to_encode = [str(ref_text)] + [text for text, is_valid in zip(cleaned, valid) if is_valid]
    emb = model.encode(texts_to_encode, convert_to_tensor=True, normalize_embeddings=True)
    ref_emb = emb[0:1]
    cand_emb = emb[1:]
    valid_scores = st_util.cos_sim(ref_emb, cand_emb).cpu().numpy().ravel().tolist()
    score_iter = iter(valid_scores)
    return [float(next(score_iter)) if is_valid else np.nan for is_valid in valid]


_specter_metric_model = None


def get_specter_metric_model():
    global _specter_metric_model
    if _specter_metric_model is None:
        from neurovlm.models import Specter
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _specter_metric_model = Specter(
            "allenai/specter2_aug2023refresh",
            adapter="adhoc_query",
            device=device,
        )
    return _specter_metric_model


def specter_cosines_to_reference(ref_text: str, candidate_texts: list[str]) -> list[float]:
    if not str(ref_text or "").strip():
        return [np.nan for _ in candidate_texts]
    cleaned = [str(text or "") for text in candidate_texts]
    if not cleaned:
        return []
    valid = [bool(text.strip()) for text in cleaned]
    scores = [np.nan for _ in cleaned]
    if not any(valid):
        return scores
    specter = get_specter_metric_model()
    payload = [{"title": str(ref_text), "summary": ""}]
    payload.extend({"title": text, "summary": ""} for text, is_valid in zip(cleaned, valid) if is_valid)
    with torch.no_grad():
        emb = specter(payload).detach().cpu()
        emb = F.normalize(emb, dim=-1)
        valid_scores = F.cosine_similarity(emb[0:1], emb[1:]).numpy().ravel().tolist()
    score_iter = iter(valid_scores)
    return [float(next(score_iter)) if is_valid else np.nan for is_valid in valid]


EVIDENCE_SIMILARITY_METRICS = {
    "minilm": lambda ref, candidates: sentence_cosines_to_reference(ref, candidates, _st_model),
    "mpnet": lambda ref, candidates: sentence_cosines_to_reference(ref, candidates, _mpnet_model),
    "specter": specter_cosines_to_reference,
}


def evidence_similarity_rows(dataset: str, sample: str, table: pd.DataFrame, full_table: pd.DataFrame, references: dict[str, str]):
    evidence = evidence_text(table)
    seed_key = f"{dataset}|{sample}|{B2T_RANDOM_EVIDENCE_SEED}".encode("utf-8")
    rng_seed = int.from_bytes(hashlib.blake2b(seed_key, digest_size=4).digest(), "little")
    rng = np.random.default_rng(rng_seed)
    random_evidences = random_evidence_texts(full_table, rng)
    candidate_evidences = [evidence] + random_evidences

    rows = []
    for ref_name, ref_text in references.items():
        row = {
            "dataset": dataset,
            "sample": sample,
            "reference": ref_name,
            "evidence_top_k": B2T_EVIDENCE_TOP_K,
            "random_baseline_n_samples": B2T_RANDOM_EVIDENCE_N_SAMPLES,
            "reference_text": str(ref_text)[:500],
            "retrieval_evidence": evidence[:1000],
        }
        for metric_name, metric_fn in EVIDENCE_SIMILARITY_METRICS.items():
            scores = metric_fn(ref_text, candidate_evidences)
            actual = float(scores[0]) if scores else np.nan
            random_arr = np.asarray(scores[1:], dtype=float)
            random_mean = float(np.nanmean(random_arr)) if len(random_arr) else np.nan
            random_std = float(np.nanstd(random_arr)) if len(random_arr) else np.nan
            row[f"{metric_name}_evidence_similarity"] = actual
            row[f"random_{metric_name}_evidence_similarity_mean"] = random_mean
            row[f"random_{metric_name}_evidence_similarity_std"] = random_std
            row[f"{metric_name}_evidence_similarity_minus_random"] = actual - random_mean
        rows.append(row)
    return rows


In [ ]:
evidence_metric_rows = []
retrieval_examples = []

for dataset, sample, latent, gold_terms, references in tqdm(list(dataset_records_for_retrieval_eval()), desc="Approach 3 evidence similarity"):
    table = retrieval_table_for_sample(latent, dataset, sample)
    full_table = full_retrieval_table_for_sample(latent, dataset, sample)
    evidence_metric_rows.extend(evidence_similarity_rows(dataset, sample, table, full_table, references))
    retrieval_examples.append({
        "dataset": dataset,
        "sample": sample,
        "gold_terms": "; ".join(gold_terms[:50]),
        "top_terms": "; ".join(table_terms(table)[:B2T_TERM_TOP_K]),
        "top_context": evidence_text(table),
    })

b2t_evidence_similarity_df = pd.DataFrame(evidence_metric_rows)
b2t_retrieval_examples_df = pd.DataFrame(retrieval_examples)
b2t_evidence_similarity_df.to_csv(OUTPUT_DIR / "b2t_approach3_retrieval_evidence_similarity.csv", index=False)
b2t_evidence_similarity_df.to_json(OUTPUT_DIR / "b2t_approach3_retrieval_evidence_similarity.json", orient="records", indent=2)
b2t_retrieval_examples_df.to_csv(OUTPUT_DIR / "b2t_approach3_retrieval_evidence_examples.csv", index=False)
b2t_retrieval_examples_df.to_json(OUTPUT_DIR / "b2t_approach3_retrieval_evidence_examples.json", orient="records", indent=2)

_evidence_summary_cols = [
    "minilm_evidence_similarity",
    "random_minilm_evidence_similarity_mean",
    "minilm_evidence_similarity_minus_random",
    "mpnet_evidence_similarity",
    "random_mpnet_evidence_similarity_mean",
    "mpnet_evidence_similarity_minus_random",
    "specter_evidence_similarity",
    "random_specter_evidence_similarity_mean",
    "specter_evidence_similarity_minus_random",
]
b2t_evidence_similarity_summary_df = b2t_evidence_similarity_df.groupby(["dataset", "reference"])[_evidence_summary_cols].agg(["mean", "std", "count"]).round(3)
b2t_evidence_similarity_summary_df.to_csv(OUTPUT_DIR / "b2t_approach3_retrieval_evidence_similarity_summary.csv")
b2t_evidence_similarity_summary_df.reset_index().to_json(OUTPUT_DIR / "b2t_approach3_retrieval_evidence_similarity_summary.json", orient="records", indent=2)
display(b2t_evidence_similarity_summary_df)


### Approach 3 Plots

These plots summarize whether retrieved evidence is closer to the reference text than random evidence sampled from the same retrieval corpus. The first plot compares actual evidence similarity with the random baseline. The second plot shows the actual-minus-random lift; values above zero mean retrieval evidence is more reference-like than chance.


In [ ]:
if len(b2t_evidence_similarity_df):
    plot_metrics = ["minilm", "mpnet", "specter"]
    plot_rows = []
    for metric in plot_metrics:
        actual_col = f"{metric}_evidence_similarity"
        random_col = f"random_{metric}_evidence_similarity_mean"
        delta_col = f"{metric}_evidence_similarity_minus_random"
        grouped = (
            b2t_evidence_similarity_df
            .groupby(["dataset", "reference"])[[actual_col, random_col, delta_col]]
            .mean()
            .reset_index()
        )
        for _, row in grouped.iterrows():
            label = f"{row['dataset']}\n{row['reference']}"
            plot_rows.append({"metric": metric, "label": label, "score_type": "actual", "score": row[actual_col]})
            plot_rows.append({"metric": metric, "label": label, "score_type": "random", "score": row[random_col]})
            plot_rows.append({"metric": metric, "label": label, "score_type": "minus_random", "score": row[delta_col]})
    approach3_plot_df = pd.DataFrame(plot_rows)
    approach3_plot_df.to_csv(OUTPUT_DIR / "b2t_approach3_plot_summary.csv", index=False)
    approach3_plot_df.to_json(OUTPUT_DIR / "b2t_approach3_plot_summary.json", orient="records", indent=2)

    labels = approach3_plot_df["label"].drop_duplicates().tolist()
    x = np.arange(len(labels))
    width = 0.36
    fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5 * len(plot_metrics), 4), sharey=False)
    if len(plot_metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, plot_metrics):
        sub = approach3_plot_df[(approach3_plot_df["metric"] == metric) & (approach3_plot_df["score_type"].isin(["actual", "random"]))]
        actual = sub[sub["score_type"] == "actual"].set_index("label").reindex(labels)["score"].to_numpy()
        random = sub[sub["score_type"] == "random"].set_index("label").reindex(labels)["score"].to_numpy()
        ax.bar(x - width / 2, actual, width, label="actual retrieved evidence")
        ax.bar(x + width / 2, random, width, label="random evidence")
        ax.set_title(f"{metric.upper()} evidence similarity")
        ax.set_xticks(x, labels, rotation=35, ha="right")
        ax.set_ylabel("Cosine similarity")
        ax.grid(axis="y", alpha=0.25)
        ax.legend(frameon=False)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_approach3_evidence_similarity_vs_random.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5 * len(plot_metrics), 4), sharey=True)
    if len(plot_metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, plot_metrics):
        sub = approach3_plot_df[(approach3_plot_df["metric"] == metric) & (approach3_plot_df["score_type"] == "minus_random")]
        delta = sub.set_index("label").reindex(labels)["score"].to_numpy()
        colors = ["#4c78a8" if v >= 0 else "#d62728" for v in delta]
        ax.bar(x, delta, color=colors)
        ax.axhline(0, color="0.35", linestyle="--", linewidth=1)
        ax.set_title(f"{metric.upper()} lift over random")
        ax.set_xticks(x, labels, rotation=35, ha="right")
        ax.set_ylabel("Actual - random mean")
        ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_approach3_evidence_similarity_minus_random.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No Approach 3 evidence rows to plot.")


In [ ]:
if RUN_GENERATED_TEXT:
    # Pre-download Hugging Face assets used by BERTScore so metric calls do not
    # repeatedly block on tokenizer/model metadata requests during the eval loop.
    def predownload_hf_model(model_name: str):
        print(f"Pre-downloading Hugging Face model for BERTScore: {model_name}")
        AutoTokenizer.from_pretrained(model_name, use_fast=False)
        AutoModel.from_pretrained(model_name)
        print("BERTScore model is cached.")

    predownload_hf_model(BERTSCORE_MODEL)
else:
    print("Skipping BERTScore model pre-download because RUN_GENERATED_TEXT = False.")


## Generated Text Evaluation

This optional section compares LLM-generated text against the ground-truth text. It is disabled by default with `RUN_GENERATED_TEXT = False` because it is much slower than the retrieval evidence metrics.

Metrics in this section, when enabled:

- `bert_p`, `bert_r`, `bert_f1`: BERTScore precision, recall, and F1 between generated text and ground truth. These check token-level contextual semantic overlap.
- `sem_sim`: MiniLM sentence-embedding similarity between generated text and ground truth.
- `nvlm_sim`: cosine similarity between the generated text embedding and the brain query in NeuroVLM's shared latent space.
- Network label accuracy metrics: for Networks, these check whether the generated text names the correct network label by alias matching or semantic matching.

Because generation depends on the LLM as well as the retrieval results, this section should be interpreted after Approach 1 and Approach 3.


In [ ]:
if RUN_GENERATED_TEXT:
    b2t_frames = []

    if RUN_NETWORKS:
        records = []
        for net_name, d in tqdm(networks_data.items(), desc="Networks B2T"):
            records.extend(run_b2t(net_name, d["latent"], d["short_gt"], d["long_gt"], SHORT_PROMPT_GENERAL, LONG_PROMPT))
        b2t_net_df = add_network_label_accuracy(pd.DataFrame(records))
        b2t_net_df["dataset"] = "networks"
        b2t_frames.append(b2t_net_df)

    if RUN_PUBMED:
        records = []
        for d in tqdm(pubmed_eval, desc="PubMed B2T"):
            records.extend(run_b2t(str(d["pmid"]), d["latent"], d["short_gt"], d["long_gt"], SHORT_PROMPT_PUBMED, LONG_PROMPT, datasets=[PUBMED_B2T_DATASET]))
        b2t_pubmed_df = pd.DataFrame(records)
        b2t_pubmed_df["dataset"] = "pubmed"
        b2t_frames.append(b2t_pubmed_df)

    if RUN_NEUROVAULT:
        records = []
        for d in tqdm(neurovault_eval, desc="NeuroVault B2T"):
            records.extend(run_b2t(str(d["doi"]), d["latent"], d["short_gt"], d["long_gt"], SHORT_PROMPT_GENERAL, LONG_PROMPT))
        b2t_nv_df = pd.DataFrame(records)
        b2t_nv_df["dataset"] = "neurovault"
        b2t_frames.append(b2t_nv_df)

    b2t_all = pd.concat(b2t_frames, ignore_index=True)
    b2t_all.to_csv(OUTPUT_DIR / "brain_to_text_metrics_v2.csv", index=False)
    b2t_all.to_json(OUTPUT_DIR / "brain_to_text_metrics_v2.json", orient="records", indent=2)
    b2t_all.head()
else:
    print("Skipping generated text evaluation because RUN_GENERATED_TEXT = False.")
    b2t_all = pd.DataFrame()


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    def recall_at_1(df: pd.DataFrame) -> float:
        if len(df) < 2:
            return np.nan
        generated = df["generated"].astype(str).tolist()
        with torch.no_grad():
            raw = nvlm._encode_text(generated)
            z_text = nvlm._proj_head_text_infonce(raw.to(nvlm.device))
            z_text = F.normalize(z_text, dim=-1).cpu()
        brain_embs = []
        for _, row in df.iterrows():
            source = row["dataset"]
            name = row["name"]
            if source == "networks":
                brain_embs.append(networks_data[name]["latent"])
            elif source == "pubmed":
                brain_embs.append(next(d["latent"] for d in pubmed_eval if str(d["pmid"]) == str(name)))
            elif source == "neurovault":
                brain_embs.append(next(d["latent"] for d in neurovault_eval if str(d["doi"]) == str(name)))
        z_brain = F.normalize(torch.stack([b.cpu() for b in brain_embs]), dim=-1)
        scores = z_text @ z_brain.T
        return float((scores.argmax(dim=1) == torch.arange(len(df))).float().mean())

    recall_rows = []
    for (dataset, mode), sub in b2t_all.groupby(["dataset", "mode"]):
        recall_rows.append({"dataset": dataset, "mode": mode, "recall_at_1": recall_at_1(sub.reset_index(drop=True)), "n": len(sub)})
    recall_df = pd.DataFrame(recall_rows)
    summary = b2t_all.groupby(["dataset", "mode"])[["nvlm_sim", "bert_f1", "sem_sim"]].agg(["mean", "std", "count"]).round(3)
    summary.to_csv(OUTPUT_DIR / "b2t_generated_text_metric_summary.csv")
    summary.reset_index().to_json(OUTPUT_DIR / "b2t_generated_text_metric_summary.json", orient="records", indent=2)
    recall_df.round(3).to_csv(OUTPUT_DIR / "b2t_generated_text_recall_at_1.csv", index=False)
    recall_df.round(3).to_json(OUTPUT_DIR / "b2t_generated_text_recall_at_1.json", orient="records", indent=2)
    display(summary)
    display(recall_df.round(3))

    if "network_label_correct" in b2t_all.columns:
        label_summary = b2t_all[b2t_all["dataset"] == "networks"].groupby("mode")["network_label_correct"].agg(["mean", "sum", "count"]).round(3)
        label_summary.to_csv(OUTPUT_DIR / "b2t_generated_text_network_label_accuracy_summary.csv")
        label_summary.reset_index().to_json(OUTPUT_DIR / "b2t_generated_text_network_label_accuracy_summary.json", orient="records", indent=2)
        display(label_summary)
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    b2t_all[(b2t_all["mode"] == "short") & (b2t_all["dataset"] == "pubmed")].iloc[0].get("generated")
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    '''
    '[kg_mesh] sim=0.534 | surgical equipment\n
     [kg_mesh] sim=0.529 | acetylene\n
     [kg_mesh] sim=0.522 | equipment and supplies, hospital\n
     [kg_mesh] sim=0.522 | hospitals, packaged\n
     [kg_mesh] sim=0.519 | cellular neural networks, computer'
    '''
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    ''''
    Acoustic neuromas, benign tumors of the auditory nerve, exhibit correlations between their morphology and 
    otoneurological manifestations. The size and site of origin of these tumors influence clinical presentation,
     with patients having lateral neuromas typically experiencing early subjective hearing loss due to smaller 
     tumors often confined to the internal auditory canal. In contrast, medial neuromas tend to be larger and 
     may grow without causing significant audiological symptoms, often preserving normal hearing function. 
     The sensitivity of the stapedial reflex test is higher for lateral neuromas, while vestibular tests 
     indicate a higher frequency of central vestibular involvement in larger tumors. The combination of 
     brainstem auditory evoked potentials and vestibular tests can effectively identify acoustic neuromas,
       offering an optimal level of sensitivity.'


    '''
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    '''
    'Surgical Equipment encompasses a range of instruments and tools essential for medical interventions, including those related to Magnetic Resonance Imaging (MRI). 
    Acetylene, though not directly involved in biological processes, plays a role in MRI by being used in the cooling systems of some MRI machines. 
    Equipment and Supplies, Hospital, refers to the diverse set of medical devices and consumables utilized in hospitals for various diagnostic procedures, 
    which can include MRI and other imaging techniques. Hospitals, Packaged, denotes pre-assembled medical equipment and supplies designed for efficient use in 
    clinical settings, often including items that support MRI and ultrasonography. These components collectively ensure the smooth operation and precision of surgical 
    and diagnostic procedures, highlighting the integral role of specialized tools and technology in modern medical practice.'
    '''
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    '''
    "Neuroimaging Evidence of Surgical Equipment Usage in MRI Procedures\n\nSurgical equipment and supplies used in hospitals are crucial for various medical 
    interventions, including those related to Magnetic Resonance Imaging (MRI). Acetylene, though not a direct component of brain activity during MRI, plays 
    a role in the technology's operation. Hospitals often package these surgical tools and supplies together, facilitating efficient use in diagnostic procedures. 
    The underlying neural network supporting such tasks might involve cellular neural networks, which model biological neurons to recognize patterns and process signals,
    reflecting the complex integration of medical equipment with brain functions during imaging procedures."
    '''
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    "'Acoustic neuroma: correlations between morphology and otoneurological manifestations.'"
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    plot_df = b2t_all.copy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metric_specs = [("nvlm_sim", "NeuroVLM latent similarity"), ("bert_f1", "BERTScore F1"), ("sem_sim", "Sentence semantic similarity")]
    for ax, (metric, title) in zip(axes, metric_specs):
        groups = [g[metric].dropna().values for _, g in plot_df.groupby("dataset")]
        labels = [k for k, _ in plot_df.groupby("dataset")]
        ax.boxplot(groups, labels=labels, showmeans=True)
        ax.set_title(title)
        ax.set_ylim(min(-0.05, np.nanmin(plot_df[metric]) - 0.05), min(1.05, np.nanmax(plot_df[metric]) + 0.1))
        ax.grid(axis="y", alpha=0.25)
        ax.tick_params(axis="x", rotation=15)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_metric_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    def _brain_latents_for_group(df):
        brain_embs = []
        for _, row in df.iterrows():
            source = row["dataset"]
            name = row["name"]
            if source == "networks":
                brain_embs.append(networks_data[name]["latent"])
            elif source == "pubmed":
                brain_embs.append(next(d["latent"] for d in pubmed_eval if str(d["pmid"]) == str(name)))
            elif source == "neurovault":
                brain_embs.append(next(d["latent"] for d in neurovault_eval if str(d["doi"]) == str(name)))
        return F.normalize(torch.stack([b.cpu() for b in brain_embs]), dim=-1)


    def _text_latents_for_group(df):
        nvlm._ensure_projection_heads()
        generated = df["generated"].astype(str).tolist()
        with torch.no_grad():
            raw = nvlm._encode_text(generated)
            z_text = nvlm._proj_head_text_infonce(raw.to(nvlm.device))
            return F.normalize(z_text, dim=-1).cpu()

    baseline_rows = []
    for (dataset, mode), sub in b2t_all.groupby(["dataset", "mode"]):
        if len(sub) < 2:
            continue
        sub = sub.reset_index(drop=True)
        z_text = _text_latents_for_group(sub)
        z_brain = _brain_latents_for_group(sub)
        scores = z_text @ z_brain.T
        eye = torch.eye(len(sub), dtype=torch.bool)
        for val in scores[eye].numpy():
            baseline_rows.append({"dataset": dataset, "mode": mode, "pair": "matched", "score": float(val)})
        for val in scores[~eye].numpy():
            baseline_rows.append({"dataset": dataset, "mode": mode, "pair": "random/off-diagonal", "score": float(val)})

    baseline_df = pd.DataFrame(baseline_rows)
    fig, ax = plt.subplots(figsize=(8, 4))
    if len(baseline_df):
        matched = baseline_df[baseline_df["pair"] == "matched"]["score"]
        random_pairs = baseline_df[baseline_df["pair"] == "random/off-diagonal"]["score"]
        bins = np.linspace(min(baseline_df["score"].min(), 0), baseline_df["score"].max(), 24)
        ax.hist(random_pairs, bins=bins, alpha=0.55, label="random/off-diagonal pairs", color="lightgray", edgecolor="white")
        ax.hist(matched, bins=bins, alpha=0.75, label="matched generated text", color="steelblue", edgecolor="white")
        for x, lab, color in [(matched.median(), "matched median", "steelblue"), (random_pairs.median(), "random median", "gray")]:
            ax.axvline(x, color=color, linestyle="--", linewidth=1)
            ax.text(x, ax.get_ylim()[1] * 0.92, f"{lab}\n{x:.3f}", ha="center", va="top", fontsize=8)
    else:
        vals = b2t_all["nvlm_sim"].dropna()
        ax.hist(vals, bins=min(20, max(5, len(vals) // 2)), alpha=0.75, color="steelblue", edgecolor="white")
        ax.text(0.5, 0.9, "Need at least two samples per group for random-pair baseline", transform=ax.transAxes, ha="center")
    ax.set_title("How to read nvlm_sim: matched text versus random pairs")
    ax.set_xlabel("NeuroVLM latent cosine similarity")
    ax.set_ylabel("Pairs")
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_nvlm_sim_scale.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    if "network_label_correct" in b2t_all.columns:
        net = b2t_all[b2t_all["dataset"] == "networks"].copy()
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        acc = net.groupby("mode")["network_label_correct"].mean()
        acc.to_csv(OUTPUT_DIR / "b2t_network_label_accuracy_by_mode.csv")
        acc.reset_index().to_json(OUTPUT_DIR / "b2t_network_label_accuracy_by_mode.json", orient="records", indent=2)
        axes[0].bar(acc.index, acc.values, color=["#4c78a8", "#72b7b2"])
        axes[0].set_ylim(0, 1)
        axes[0].set_ylabel("Accuracy")
        axes[0].set_title("Network label accuracy")
        axes[0].grid(axis="y", alpha=0.25)

        cm = pd.crosstab(net["true_network_key"], net["pred_network_key"], normalize="index")
        cm.to_csv(OUTPUT_DIR / "b2t_network_label_confusion_matrix.csv")
        cm.reset_index().to_json(OUTPUT_DIR / "b2t_network_label_confusion_matrix.json", orient="records", indent=2)
        im = axes[1].imshow(cm.values, vmin=0, vmax=1, cmap="Blues")
        axes[1].set_xticks(range(len(cm.columns)), cm.columns, rotation=45, ha="right")
        axes[1].set_yticks(range(len(cm.index)), cm.index)
        axes[1].set_title("Predicted network label by true label")
        fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        fig.tight_layout()
        plt.savefig(OUTPUT_DIR / "b2t_network_label_accuracy.png", dpi=150, bbox_inches="tight")
        plt.show()

        network_label_details_df = net[["name", "mode", "true_network_key", "pred_network_key", "network_label_correct", "label_match_method", "generated"]]
        network_label_details_df.to_csv(OUTPUT_DIR / "b2t_network_label_accuracy_details.csv", index=False)
        network_label_details_df.to_json(OUTPUT_DIR / "b2t_network_label_accuracy_details.json", orient="records", indent=2)
        display(network_label_details_df)
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")
